In [ ]:
library(Seurat)
library(pheatmap)
library(RColorBrewer)
library(ComplexHeatmap)
library(tidyr)
library(ggplot2)
library(dplyr)
library(cowplot)
library(data.table)
library(harmony)
library(ggpubr)
library(ggpmisc)
set.seed(123)
library(tidyverse)
library(ggsci)
library(patchwork)
library(rhdf5)
library(future)
options(future.globals.maxSize = 50 * 1024^3)
plan(multisession, workers=16)
library(viridis)
library(ggrepel)

In [ ]:
output_pdf = './03.corrleation_result/'

In [ ]:
cols_celltype = c("#DC143C","#0000FF","#20B2AA","#FFA500","#9370DB","#98FB98","#E8E8E8")
names(cols_celltype) = c('EPI','PE/HYPO','TE','INTER','NR','ICM','Other')
cols_species = c("#DC143C","#0000FF","#20B2AA","#FFA500","#9370DB","#98FB98","#F08080","#1E90FF","#E8E8E8")
names(cols_species) = c('human_blastoid','human_blastocyst_E5','human_blastocyst_E6','human_blastocyst_E7',
                        'chimpanzee_blastoid','macaque_blastocyst','Orangutan_blastoid','Orangutan_bi_blastoid',
                        'Other')

In [ ]:

######################read data####################


In [ ]:
Data_all = readRDS(paste0('./01.integration_result/',"/","CrossSpecies_CCA.rds"))

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)
p = DimPlot(Data_all, reduction = "umap.cca",label = FALSE,group.by= "label",cols=c(cols_species),pt.size =0.5)+coord_fixed()+ 
    theme_minimal()+theme(panel.grid.major = element_blank(), 
        panel.grid.minor = element_blank(), 
        panel.border = element_blank(), 
        axis.title = element_blank(),  
        axis.text = element_blank())+
  theme()+labs(title = "")
print(p)

In [ ]:

######################corrleation analysis in different species#####################


In [ ]:
#different species
Date_test = Data_all
expr = AggregateExpression(Date_test, group.by = "label", assays = "RNA", slot = "data")$RNA
#common gene
common_genes = rownames(expr)[
  apply(expr, 1, function(x) all(x > 0))
]
print(length(common_genes))
expr = expr[common_genes, ]
cor_mat = cor(as.matrix(expr), method = "pearson")
options(repr.plot.width=6, repr.plot.height=6)
p = pheatmap(cor_mat,
         clustering_distance_rows = "correlation",
         clustering_distance_cols = "correlation",
         main = "",display_numbers = TRUE)
print(p)
pdf(file = paste0(output,'/',"Heatmap_correlation_species_common_gene.pdf"), width = 6, height = 6)
print(p)
dev.off()

In [ ]:
#different species
Date_test = Data_all
emb = Embeddings(Date_test, reduction = "integrated.cca")[, 1:30] 
emb_df = as.data.frame(emb)
emb_df$label = Date_test$label  
centroids = emb_df %>%
  group_by(label) %>%
  summarise(across(starts_with("integratedcca"), mean))
centroid_matrix = as.matrix(centroids[, -1])
dist_mat = as.matrix(dist(centroid_matrix))
rownames(dist_mat) = centroids$label
colnames(dist_mat) = centroids$label
p = pheatmap(dist_mat, 
         main = "",  #Species centroid distance
         clustering_distance_rows = "euclidean", 
         clustering_distance_cols = "euclidean",display_numbers = TRUE)
print(p)
pdf(file = paste0(output,'/',"Heatmap_species_UMAP_centroid_distance.pdf"), width = 6, height = 6)
print(p)
dev.off()

In [ ]:

######################corrleation analysis in different species and celltype#####################


In [ ]:
#different species and celltype
Date_test = subset(Data_all, subset = !label %in% c('macaque_blastocyst'))
Date_test$test = paste0(Date_test$New_ann,'_',Date_test$label)
#get Embeddings
emb = Embeddings(Date_test, reduction = "integrated.cca")
species_avg = sapply(
  unique(Date_test$test),
  function(sp)
    colMeans(emb[Date_test$test == sp, , drop = FALSE])
)
cor_mat = cor(species_avg, method = "pearson")
options(repr.plot.width=10, repr.plot.height=10)
p = pheatmap(cor_mat,
         clustering_distance_rows = "correlation",
         clustering_distance_cols = "correlation",
         main = "",display_numbers = TRUE)
print(p)
pdf(file = paste0(output_pdf,'/',"Heatmap_correlation_species2celltype_Embeddings.pdf"), width = 10, height = 10)
print(p)
dev.off()

In [ ]:

######################PCA analysis in different species#####################


In [ ]:

####################All species###########################


In [ ]:
Date_subset = Data_all
expr = AggregateExpression(Date_subset, group.by = "label", assays = "RNA", slot = "data")$RNA
colnames(expr) = gsub("-","_",colnames(expr))
common_genes = VariableFeatures(Date_subset)
print(length(common_genes))
expr = expr[common_genes, ]
expr = as.matrix(expr)
pca.res = prcomp(t(expr),  scale. = FALSE)
pca.df = as.data.frame(pca.res$x[, 1:2])
pca.df$label = rownames(pca.df)
pca.df$PC1_scaled = scale(pca.df$PC1)
pca.df$PC2_scaled = scale(pca.df$PC2)

In [ ]:
options(repr.plot.width=6, repr.plot.height=4)
p = ggplot(pca.df, aes(x = PC1, y = PC2, color = label)) +
  geom_point(size = 4) +
  geom_text_repel(aes(label = label), size = 4) +
  theme_classic() +
  labs(
    x = paste0("PC1 (", round(summary(pca.res)$importance[2,1] * 100, 1), "%)"),
    y = paste0("PC2 (", round(summary(pca.res)$importance[2,2] * 100, 1), "%)")
  ) +scale_color_manual(values = cols)+
  scale_x_continuous(labels = scales::label_number()) +
scale_y_continuous(labels = scales::label_number())
print(p)
pdf(file = paste0(output_pdf,'/',"PCA.pdf"), width = 6, height = 6)
print(p)
dev.off()

In [ ]:
pca.mat = as.matrix(pca.df[, c("PC1", "PC2")])
rownames(pca.mat) = pca.df$label 
cor_mat = cor(t(pca.mat), method = "pearson") 
options(repr.plot.width=8, repr.plot.height=8)
p = pheatmap(cor_mat,
         clustering_distance_rows = "correlation",
         clustering_distance_cols = "correlation",
         main = "Correlation between different species",display_numbers = FALSE)
print(p)
pdf(file = paste0(output_pdf,'/',"Heatmap_pca.pdf"), width = 6, height = 6)
print(p)
dev.off()
print(p)

In [ ]:

####################noMacaqueBlastocyst###########################


In [ ]:
Date_subset = Data_all
Date_subset = subset(Data_all, subset = !label %in% c('macaque_blastocyst'))
expr = AggregateExpression(Date_subset, group.by = "label", assays = "RNA", slot = "data")$RNA
colnames(expr) = gsub("-","_",colnames(expr))
common_genes = VariableFeatures(Date_subset)
print(length(common_genes))
expr = expr[common_genes, ]
expr = as.matrix(expr)
pca.res = prcomp(t(expr),  scale. = FALSE)
pca.df = as.data.frame(pca.res$x[, 1:2])
pca.df$label = rownames(pca.df)
pca.df$PC1_scaled = scale(pca.df$PC1)
pca.df$PC2_scaled = scale(pca.df$PC2)

In [ ]:
options(repr.plot.width=6, repr.plot.height=4)
p = ggplot(pca.df, aes(x = PC1_scaled, y = PC2_scaled, color = label)) +
  geom_point(size = 4) +
  geom_text_repel(aes(label = label), size = 4) +
  theme_classic() +
  labs(
    x = paste0("PC1 (", round(summary(pca.res)$importance[2,1] * 100, 1), "%)"),
    y = paste0("PC2 (", round(summary(pca.res)$importance[2,2] * 100, 1), "%)")
  ) +scale_color_manual(values = cols)+
  scale_x_continuous(labels = scales::label_number()) +
scale_y_continuous(labels = scales::label_number())
print(p)
pdf(file = paste0(output_pdf,'/',"PCA_noMacaque.pdf"), width = 6, height = 6)
print(p)
dev.off()

In [ ]:
pca.mat = as.matrix(pca.df[, c("PC1", "PC2")])
rownames(pca.mat) = pca.df$label 
cor_mat = cor(t(pca.mat), method = "pearson") 
options(repr.plot.width=8, repr.plot.height=8)
p = pheatmap(cor_mat,
         clustering_distance_rows = "correlation",
         clustering_distance_cols = "correlation",
         main = "Correlation between different species",display_numbers = FALSE)
print(p)
pdf(file = paste0(output_pdf,'/',"Heatmap_pca_noMacaque.pdf"), width = 6, height = 6)
print(p)
dev.off()
print(p)